# qlib benchmark run — free GPU (Kaggle / Colab)

Runs `scripts/gpu_run.py` against the private `Riyu-D1/quant-research` repo.

**Setup once:** store a GitHub PAT (read+write on the repo) as secret `GH_TOKEN`
- Kaggle: *Add-ons → Secrets*
- Colab: *key icon → Secrets*

Enable a GPU accelerator before running: Kaggle *Settings → Accelerator (T4/P100)*, Colab *Runtime → Change runtime type → T4*.

US models (`*_us`) need the Alpaca-built qlib dump, published as workflow
artifact `us-qlib-dump` by the `us-alpaca-data` GitHub Action. The fetch cell
below downloads it via the GitHub API using `GH_TOKEN` — **no Alpaca keys in
this notebook**.

In [ ]:
import os
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
WORK = '/kaggle/working' if ON_KAGGLE else ('/content' if ON_COLAB else os.path.expanduser('~/cloud-run'))
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local', '| workdir:', WORK)

def get_secret(name):
    """GH_TOKEN etc. — Kaggle secrets, Colab userdata, or plain env var."""
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try:
            return UserSecretsClient().get_secret(name)
        except Exception:
            return os.environ.get(name)
    if ON_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return os.environ.get(name)
    return os.environ.get(name)


In [ ]:
GH_TOKEN = get_secret('GH_TOKEN')
if not GH_TOKEN:
    raise RuntimeError('Set GH_TOKEN secret (PAT with repo access)')
os.environ['GH_TOKEN'] = GH_TOKEN   # phase4 news scorer reads it from env
REPO_URL = f'https://{GH_TOKEN}@github.com/Riyu-D1/quant-research.git'
BRANCH = 'phase4/adaptive'   # phase-4 code (phase4_news_score) lives here
if not os.path.exists('quant-research'):
    !git clone --branch $BRANCH $REPO_URL
# idempotent: re-running cells after a kernel restart must not fail on %cd
if os.path.basename(os.getcwd()) != 'quant-research':
    %cd quant-research

In [ ]:
!pip -q install uv
!uv python install 3.11
!uv sync --frozen --no-dev
import sys, subprocess
print(subprocess.run(['.venv/bin/python', '-c',
    'import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())'],
    capture_output=True, text=True).stdout)

In [ ]:
# Fetch the US qlib dump (artifact from the us-alpaca-data workflow) when
# any *_us model is requested. Uses GH_TOKEN against the GitHub API —
# works on a private repo where `gh run download` would also need auth.
US_DUMP = 'data/qlib/us_alpaca'
# Phase-2 names end in _wf (walk-forward OOF preds into results/oof/<mkt>/);
# phase1 names (transformer_cn hist_cn transformer_us) still work too.
# PHASE 6: one combined GPU session — the four remaining published
# CN walk-forward models (feat20 FilterCol specs in the yaml) plus
# TCN on the US grid. --train-holdout inside each registry entry
# trains OOF folds AND the holdout fold in the same pass.
MODELS = 'tcn_cn_wf alstm_cn_wf tra_cn_wf igmtf_cn_wf tcn_us_wf'  # edit me
# US entries are marked by an '_us' component anywhere in the name —
# transformer_us, hist_us_wf, hist_us_ho, phase2_us — not just a suffix.
if any('_us' in m for m in MODELS.split()) \
        and not os.path.exists(f'{US_DUMP}/calendars/day.txt'):
    import io, json, zipfile, urllib.request
    api = 'https://api.github.com/repos/Riyu-D1/quant-research'
    hdr = {'Authorization': f'Bearer {GH_TOKEN}',
           'Accept': 'application/vnd.github+json'}
    req = urllib.request.Request(
        f'{api}/actions/artifacts?name=us-qlib-dump&per_page=100', headers=hdr)
    arts = [a for a in json.load(urllib.request.urlopen(req))['artifacts']
            if not a['expired']]
    if not arts:
        raise RuntimeError('no us-qlib-dump artifact — run the '
                           'us-alpaca-data workflow first')
    url = sorted(arts, key=lambda a: a['created_at'])[-1]['archive_download_url']
    # urllib re-sends Authorization on the blob-storage redirect → 401; requests drops it cross-host
    import requests
    r = requests.get(url, headers=hdr, timeout=600)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    os.makedirs(US_DUMP, exist_ok=True)
    z.extractall(US_DUMP)
    print('extracted us-qlib-dump ->', US_DUMP)
else:
    print('us dump present or not needed')

In [ ]:
!.venv/bin/python scripts/gpu_run.py --list

In [ ]:
# `!cmd` prints output but never raises — a failed run would silently
# fall through to the push cell. subprocess(check=True) raises
# CalledProcessError and the notebook STOPS here on failure.
import subprocess
subprocess.run(['.venv/bin/python', 'scripts/gpu_run.py', '--models']
               + MODELS.split(), check=True)


In [ ]:
!cat results/LEADERBOARD.md 2>/dev/null; ls -la results/runs/

In [ ]:
# optional: push results back to a dated branch (set PUSH_RESULTS=1)
import os, datetime
if (os.environ.get('PUSH_RESULTS') == '1' or get_secret('PUSH_RESULTS') == '1') and GH_TOKEN:
    br = f"results/{'kaggle' if ON_KAGGLE else 'colab'}-{datetime.date.today()}"
    !git checkout -b $br
    # scoped adds ONLY — never `git add results/` wholesale: a run that
    # clobbers results/LEADERBOARD.md must not get staged.
    !git add results/news_sentiment results/runs results/oof EXPERIMENTS.md 2>/dev/null || true
    !git -c user.name='gpu-run' -c user.email='noreply@users.noreply.github.com' commit -m 'gpu run results' || true
    !git push origin $br
    print('pushed branch', br)
else:
    print('PUSH_RESULTS not set — results stay in the session; download results/runs/*.json manually')
